In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 1. Initialize the LLM (Temperature 0.0 for maximum determinism)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# 2. The Magic Method: Binding the Pydantic Schema to the Model
# This creates a new, constrained version of the LLM that ONLY outputs our Graph object.
graph_extractor = llm.with_structured_output(KnowledgeGraph)

# 3. The Instruction Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an elite Data Engineer extracting Knowledge Graphs from financial text. "
               "Extract all entities and relationships. Ensure source and target node IDs match exactly."),
    ("human", "{text}")
])

# 4. Building the Chain
extraction_pipeline = prompt | graph_extractor

# --- THE STRESS TEST ---
unstructured_text = """
On Friday, Microsoft's CEO Satya Nadella announced the stunning $2 Billion acquisition 
of the generative AI startup Alpha AI. The deal was highlighted in Tony's Q3 Compliance Report.
"""

print("[System] Passing unstructured text to the Constrained LLM...")

# Execute the pipeline
extracted_graph = extraction_pipeline.invoke({"text": unstructured_text})

# Prove that it is a native Python object, NOT a string.
print("\n[System] Extraction Complete. Data Type:", type(extracted_graph))
print("\n--- NODES ---")
for node in extracted_graph.nodes:
    print(f"ID: {node.id} | Label: {node.label}")

print("\n--- EDGES ---")
for edge in extracted_graph.edges:
    print(f"{edge.source_node_id} --[{edge.relationship_type}]--> {edge.target_node_id}")


Master Prompt Implementation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Notice the uncompromising, dictatorial tone of the System Prompt.
system_instruction = """
You are an elite, deterministic Information Extraction AI built for AlphaFund.
Your sole purpose is to convert unstructured financial text into a strict Knowledge Graph.

=== CORE DIRECTIVES ===
1. CANONICAL RESOLUTION: You must resolve all pronouns (he, she, it, they) and aliases ("the tech giant", "the startup") to their exact, canonical entity name. Never output a pronoun as a Node ID.
2. IMPLICIT EDGES: Infer relationships from possessives and context (e.g., "Apple's Tim Cook" implies Tim Cook WORKS_FOR Apple).
3. ONTOLOGY LOCK: You may ONLY use the relationship types defined in your schema. Do not invent new verbs.
4. NO ORPHANS: Every node you extract MUST be connected to at least one edge. Do not extract isolated facts.

=== FEW-SHOT CALIBRATION ===
Text: "Tony's Q3 report covered the Alpha AI buyout by Microsoft. Satya Nadella, its CEO, approved the $2B deal."

Reasoning Trace:
- "Tony's Q3 report" -> Tony (Executive) WROTE "Q3 report" (Document).
- "covered the Alpha AI buyout by Microsoft" -> "Q3 report" MENTIONS "Alpha AI" and "Microsoft". Microsoft ACQUIRED Alpha AI.
- "Satya Nadella, its CEO" -> "its" resolves to Microsoft. Satya Nadella WORKS_FOR Microsoft.

Target JSON Output:
{{
  "nodes": [
    {{"id": "Tony", "label": "Executive", "original_text": "Tony"}},
    {{"id": "Q3 Report", "label": "Document", "original_text": "Q3 report"}},
    {{"id": "Alpha AI", "label": "Company", "original_text": "Alpha AI"}},
    {{"id": "Microsoft", "label": "Company", "original_text": "Microsoft"}},
    {{"id": "Satya Nadella", "label": "Executive", "original_text": "Satya Nadella"}}
  ],
  "edges": [
    {{"source_node_id": "Tony", "target_node_id": "Q3 Report", "relationship_type": "WROTE"}},
    {{"source_node_id": "Q3 Report", "target_node_id": "Microsoft", "relationship_type": "MENTIONS"}},
    {{"source_node_id": "Q3 Report", "target_node_id": "Alpha AI", "relationship_type": "MENTIONS"}},
    {{"source_node_id": "Microsoft", "target_node_id": "Alpha AI", "relationship_type": "ACQUIRED"}},
    {{"source_node_id": "Satya Nadella", "target_node_id": "Microsoft", "relationship_type": "WORKS_FOR"}}
  ]
}}
"""

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", "Process the following text and return the graph:\n\n{text}")
])
